In [16]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,r2_score
import numpy as np
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv')

In [17]:
df.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [18]:
df.shape

(506, 14)

In [19]:

len(scores)

5

# Hold-Out Validation
- Here we split the original data into:
- 80% → Training
- 20% → Test
- From the 80% training data, take some portion as validation.

In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Load data
df = pd.read_csv(
    'https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv'
)

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

# Further split training data into training + validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=2
)

# Train
model = LinearRegression()
model.fit(X_train, y_train)

# Validate
y_val_pred = model.predict(X_val)

print("Validation MSE:",
      mean_squared_error(y_val, y_val_pred))

print("Validation R²:",
      r2_score(y_val, y_val_pred))

# Final test evaluation
y_test_pred = model.predict(X_test)

print("Test MSE:",
      mean_squared_error(y_test, y_test_pred))

print("Test R²:",
      r2_score(y_test, y_test_pred))

Validation MSE: 19.89114803204398
Validation R²: 0.7171746714621288
Test MSE: 19.031549663364803
Test R²: 0.772512287379061


# 2. LOOCV

In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_score
from sklearn.linear_model import LinearRegression

# Load data
df = pd.read_csv(
    'https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv'
)

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Separate test data first
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

model = LinearRegression()

# LOOCV only on training data
loo = LeaveOneOut()

scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=loo,
    scoring='neg_mean_squared_error'
)

mse_scores = -scores

print("Mean CV MSE:", mse_scores.mean())

# Train final model on ALL training data
model.fit(X_train, y_train)

# Final evaluation on untouched test data
test_pred = model.predict(X_test)

from sklearn.metrics import mean_squared_error, r2_score

print("Test MSE:", mean_squared_error(y_test, test_pred))
print("Test R²:", r2_score(y_test, test_pred))

Mean CV MSE: 25.419094631759037
Test MSE: 18.49542012244846
Test R²: 0.7789207451814409


# 3. K-Fold Cross-Validation

In [22]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression

df = pd.read_csv(
    'https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv'
)

X = df.iloc[:, :-1]
y = df.iloc[:, -1]

# Separate test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2
)

model = LinearRegression()

# 5-Fold CV on training data
kfold = KFold(n_splits=5, shuffle=True, random_state=2)

scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=kfold,
    scoring='neg_mean_squared_error'
)

mse_scores = -scores

print("MSE for each fold:", mse_scores)
print("Mean CV MSE:", mse_scores.mean())

# Train final model
model.fit(X_train, y_train)

# Test
test_pred = model.predict(X_test)

from sklearn.metrics import mean_squared_error, r2_score

print("Test MSE:", mean_squared_error(y_test, test_pred))
print("Test R²:", r2_score(y_test, test_pred))

MSE for each fold: [19.89114803 37.12110074 20.50417523 25.10482102 24.05343607]
Mean CV MSE: 25.334936219626734
Test MSE: 18.49542012244846
Test R²: 0.7789207451814409


# 4. Stratified K-Fold

- Stratified K-Fold is normally used for classification, because it preserves the proportion of each class.


In [23]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Load Iris
iris = load_iris()

X = iris.data
y = iris.target

# Separate test data first
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=2,
    stratify=y
)

# Logistic Regression
model = LogisticRegression(max_iter=1000)

# Stratified K-Fold
skfold = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=2
)

# CV ONLY on training data
scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=skfold,
    scoring='accuracy'
)

print("Fold accuracies:", scores)
print("Mean CV accuracy:", scores.mean())

# Train on complete training data
model.fit(X_train, y_train)

# Final evaluation on untouched test data
print("Test accuracy:", model.score(X_test, y_test))

Fold accuracies: [0.95833333 1.         0.95833333 0.91666667 1.        ]
Mean CV accuracy: 0.9666666666666668
Test accuracy: 1.0
